In [ ]:

import ollama
import chromadb
import pypdf
import gradio

print("All required libraries imported successfully!")

In [ ]:
!apt-get update -qq
!apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for i in range(30):
    try:
        response = requests.get("http://127.0.0.1:11434")
        if response.status_code == 200:
            print("Ollama server is running successfully!")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    print("Server failed to start. Please check.")

In [ ]:
!nvidia-smi

In [ ]:
!ollama pull gemma3:1b

In [ ]:
!ollama pull embeddinggemma

In [ ]:
!ollama list

In [ ]:
!ps aux | grep '[o]llama serve'

In [ ]:
import subprocess
import time
import requests

# Stop any previous Ollama server
subprocess.run(
    ["pkill", "-f", "ollama serve"],
    capture_output=True
)

time.sleep(2)

# Start Ollama again
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

# Wait for server
for i in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:11434",
            timeout=2
        )

        if response.status_code == 200:
            print("SUCCESS! Ollama server is running.")
            break

    except requests.exceptions.RequestException:
        time.sleep(2)

else:
    print("Ollama failed to start. Check server logs.")

In [ ]:
!ollama list

In [ ]:
import ollama

try:
    client = ollama.Client(host='http://127.0.0.1:11434')
    # List available models via the SDK
    models_list = client.list()
    print("Successfully connected to Ollama!")
    print("Available models in SDK:")
    for model in models_list.get('models', []):
        print(f" - {model.get('model')}")

    # Try a quick test run to verify inference works
    print("\nRunning a quick test generation with gemma3:1b...")
    response = client.generate(model='gemma3:1b', prompt='Hello! Say "Ollama is ready!"')
    print(f"Response: {response.get('response')}")
except Exception as e:
    print(f"Error communicating with Ollama: {e}")

In [ ]:
import ollama

try:
    client = ollama.Client(host='http://127.0.0.1:11434')
    # List available models via the SDK
    models_list = client.list()
    print("Successfully connected to Ollama!")
    print("Available models in SDK:")
    for model in models_list.get('models', []):
        print(f" - {model.get('model')}")

    # Try a quick test run to verify inference works
    print("\nRunning a quick test generation with gemma3:1b...")
    response = client.generate(model='gemma3:1b', prompt='Hello! Say "Ollama is ready!"')
    print(f"Response: {response.get('response')}")
except Exception as e:
    print(f"Error communicating with Ollama: {e}")

### Investigating Ollama Server Status
Let's check if the server is still running in the background and read its logs to diagnose the connection failure.

In [ ]:
import subprocess

# 1. Check if the process is active
ps_output = subprocess.run(["ps", "aux"], capture_output=True, text=True)
ollama_processes = [line for line in ps_output.stdout.split('\n') if 'ollama' in line]

print("--- Active Ollama Processes ---")
if ollama_processes:
    for proc in ollama_processes:
        print(proc)
else:
    print("No active Ollama process found!")

# 2. Let's start the server in the foreground briefly and capture the logs to see the error
print("\n--- Capturing Ollama Server Startup Logs ---")
try:
    # Run for 3 seconds to capture any startup errors
    proc = subprocess.run(["ollama", "serve"], capture_output=True, text=True, timeout=3)
    print("STDOUT:", proc.stdout)
    print("STDERR:", proc.stderr)
except subprocess.TimeoutExpired as e:
    # If it didn't exit in 3 seconds, it is running or hanging
    print("Process did not crash immediately. Startup Stderr:")
    print(e.stderr if e.stderr else "No immediate stderr logs.")

### Cleaning Up and Restarting Ollama
We will force kill any remaining zombie or defunct processes, launch the background server cleanly, and verify the client connection.

In [ ]:
import subprocess
import time
import requests
import ollama

# 1. Forcefully kill any running, zombie, or orphaned ollama processes
print("Cleaning up existing processes...")
subprocess.run(["pkill", "-9", "-f", "ollama"], capture_output=True)
time.sleep(2)

# 2. Launch a completely fresh Ollama server instance in the background
print("Starting fresh background Ollama server...")
server_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 3. Wait and poll the server until it is fully active
for i in range(15):
    try:
        response = requests.get("http://127.0.0.1:11434", timeout=2)
        if response.status_code == 200:
            print("SUCCESS: Ollama server started cleanly and is fully responsive!")
            break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    print("ERROR: Server failed to become responsive.")

# 4. Test communication with the python client SDK
try:
    client = ollama.Client(host='http://127.0.0.1:11434')
    models = client.list()
    print("\nConnection established! Available models:")
    for m in models.get('models', []):
        print(f" - {m.get('model')}")
except Exception as e:
    print(f"Connection test failed: {e}")

In [ ]:
import subprocess
import time
import requests

# Stop any previous Ollama server
subprocess.run(
    ["pkill", "-f", "ollama serve"],
    capture_output=True
)

time.sleep(2)

# Start Ollama again
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

# Wait for server
for i in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:11434",
            timeout=2
        )

        if response.status_code == 200:
            print("SUCCESS! Ollama server is running.")
            break

    except requests.exceptions.RequestException:
        time.sleep(2)

else:
    print("Ollama failed to start. Check server logs.")

In [ ]:
!ollama list

In [ ]:
import ollama

client = ollama.Client(
    host="http://127.0.0.1:11434"
)

response = client.chat(
    model="gemma3:1b",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in three simple sentences."
        }
    ]
)

print(response["message"]["content"])

In [ ]:
response = client.embed(
    model="embeddinggemma",
    input="RAG combines document retrieval with language models."
)

embedding = response["embeddings"][0]

print("Embedding successful!")
print("Embedding dimensions:", len(embedding))

In [ ]:

import chromadb
import uuid

# Initialize ChromaDB
chroma_client = chromadb.Client()

# Create collection
try:
    chroma_client.delete_collection("college_rag")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="college_rag",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB initialized successfully!")


In [ ]:

from pypdf import PdfReader
from google.colab import files

def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"

    return text


def split_into_chunks(text, chunk_size=800, overlap=150):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


print("PDF processing functions are ready!")


In [ ]:

uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))

text = extract_text(pdf_path)

chunks = split_into_chunks(text)

print("PDF uploaded successfully!")
print("Total characters:", len(text))
print("Total chunks:", len(chunks))

print("\nDocument preview:")
print(chunks[0][:500] if chunks else "No readable text found")


In [ ]:

import uuid

# Check whether PDF chunks exist
if not chunks:
    raise ValueError("No PDF chunks found. Please process your syllabus first.")

print("Total chunks:", len(chunks))

# Generate embeddings in batches
all_embeddings = []

for i in range(0, len(chunks), 16):

    batch = chunks[i:i + 16]

    response = client.embed(
        model="embeddinggemma",
        input=batch
    )

    batch_embeddings = response["embeddings"]

    all_embeddings.extend(batch_embeddings)

    print(
        f"Processed {min(i + 16, len(chunks))}/{len(chunks)} chunks"
    )

# Verify embedding count
assert len(all_embeddings) == len(chunks)

# Create unique IDs
ids = [str(uuid.uuid4()) for _ in chunks]

# Store in ChromaDB
collection.add(
    ids=ids,
    documents=chunks,
    embeddings=all_embeddings
)

print("\nSUCCESS!")
print("Total documents stored:", collection.count())
print("Embedding dimensions:", len(all_embeddings[0]))


In [ ]:
import subprocess
import time
import requests

# Start Ollama server
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait until server is available
for i in range(30):
    try:
        r = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=3
        )

        if r.status_code == 200:
            print("Ollama server restarted successfully!")
            print(r.json())
            break

    except requests.exceptions.RequestException:
        time.sleep(2)

else:
    raise RuntimeError("Ollama server could not start")

In [ ]:
import ollama

client = ollama.Client(
    host="http://127.0.0.1:11434",
    timeout=300
)

print("Ollama client connected!")

In [ ]:
test = client.embed(
    model="embeddinggemma",
    input="VTU syllabus and engineering subjects"
)

print("Embedding successful!")
print("Dimensions:", len(test["embeddings"][0]))

In [ ]:
import time

all_embeddings = []

for i in range(0, len(chunks), 4):

    batch = chunks[i:i+4]

    for attempt in range(3):
        try:
            result = client.embed(
                model="embeddinggemma",
                input=batch
            )

            all_embeddings.extend(result["embeddings"])
            break

        except Exception as e:
            print("Retrying batch:", i, e)

            time.sleep(5)

    else:
        raise RuntimeError(
            f"Embedding failed at chunk {i}. Please restart Ollama."
        )

    print(
        f"Completed: {len(all_embeddings)}/{len(chunks)}"
    )

print("All embeddings generated!")

In [ ]:
import uuid

collection.add(
    ids=[str(uuid.uuid4()) for _ in chunks],
    documents=chunks,
    embeddings=all_embeddings
)

print("Successfully stored VTU syllabus!")
print("Total vectors:", collection.count())

In [ ]:


def rag_answer(question):

    query_result = client.embed(
        model="embeddinggemma",
        input=question
    )

    query_embedding = query_result["embeddings"][0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=6
    )

    retrieved_chunks = results["documents"][0]

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
    You are an expert VTU academic syllabus assistant.

    Answer the student's question accurately using
    the supplied syllabus context.

    IMPORTANT RULES:
    1. Answer only what the student asks.
    2. Do not mix subject names with module topics.
    3. Do not include textbook references unless asked.
    4. Do not repeat information.
    5. Organize answers using clear headings and bullet points.
    6. If the exact semester information is unavailable,
       clearly mention that.
    7. Never invent subject names or syllabus details.

    SYLLABUS CONTEXT:
    {context}

    STUDENT QUESTION:
    {question}

    Give a clear and concise answer.
    """

    response = client.chat(
        model="gemma3:1b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response["message"]["content"]


In [ ]:
print(rag_answer(
    "List only the subject names in the first semester."
))


In [ ]:
print(rag_answer(
    "Explain the topics covered in Module 1 of Operating Systems."
))

In [ ]:
print(rag_answer(
    "What textbooks are recommended for the first semester?"
))

In [ ]:
!pip -q install gradio

In [ ]:

import gradio as gr
import uuid

# -----------------------------
# PDF Upload Function
# -----------------------------

def upload_pdf(pdf_file):

    global collection

    if pdf_file is None:
        return "Please upload a PDF document."

    try:
        text = extract_text(pdf_file)

        if not text.strip():
            return "No readable text found in PDF."

        new_chunks = split_into_chunks(text)

        if not new_chunks:
            return "No text chunks created."

        # Generate embeddings
        embeddings = []

        for i in range(0, len(new_chunks), 4):

            batch = new_chunks[i:i+4]

            result = client.embed(
                model="embeddinggemma",
                input=batch
            )

            embeddings.extend(result["embeddings"])

        # Replace old collection
        try:
            chroma_client.delete_collection("college_rag")
        except Exception:
            pass

        collection = chroma_client.create_collection(
            name="college_rag",
            metadata={"hnsw:space": "cosine"}
        )

        # Store document chunks
        collection.add(
            ids=[str(uuid.uuid4()) for _ in new_chunks],
            documents=new_chunks,
            embeddings=embeddings
        )

        return (
            "SUCCESS! VTU syllabus uploaded and processed.\n"
            f"Total chunks indexed: {len(new_chunks)}"
        )

    except Exception as e:
        return f"Upload error: {str(e)}"


# -----------------------------
# Chat Function
# -----------------------------

def chat_with_vtu(message, history):

    if not message.strip():
        return "Please enter a question."

    if collection.count() == 0:
        return "Please upload and process your syllabus first."

    try:
        return rag_answer(message)

    except Exception as e:
        return f"Chat error: {str(e)}"


# -----------------------------
# Gradio Interface
# -----------------------------

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="VTU Syllabus Assistant"
) as demo:

    gr.Markdown(
        """
        # 🎓 VTU Syllabus Assistant

        ### Your AI-powered academic companion

        Ask questions about your VTU syllabus
        using Retrieval Augmented Generation.

        **Powered by Ollama | Gemma | EmbeddingGemma | ChromaDB**
        """
    )

    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown("### 📄 Upload Syllabus")

            pdf_input = gr.File(
                label="Upload VTU PDF",
                file_types=[".pdf"],
                type="filepath"
            )

            upload_button = gr.Button(
                "Process Syllabus",
                variant="primary"
            )

            upload_status = gr.Textbox(
                label="Processing Status",
                lines=3
            )

        with gr.Column(scale=2):

            gr.Markdown("### 💬 Chat with your syllabus")

            chatbot = gr.ChatInterface(
                fn=chat_with_vtu,
                title="Ask Your VTU Assistant",
                description=(
                    "Ask about subjects, modules, "
                    "course outcomes and textbooks."
                ),
                examples=[
                    "What subjects are included in the first semester?",
                    "Explain the Operating Systems syllabus.",
                    "What are the recommended textbooks?",
                    "What topics are covered in Module 1?"
                ]
            )

    upload_button.click(
        fn=upload_pdf,
        inputs=pdf_input,
        outputs=upload_status
    )

    gr.Markdown(
        """
        ---
        **VTU Syllabus Assistant | RAG-Based Academic Chatbot**

        Built using Python, Ollama, Gemma, ChromaDB and Gradio.
        """
    )


demo.launch(share=True)
